# Notebook 07 — Data Storytelling & Communication

**Projet :** Prediction du risque d'abandon scolaire  
**Equipe :** Hugo RAGUIN · Amine TALEB · Elliot FIORESE  

Ce notebook produit la **restitution interactive** des resultats sous forme de dashboard Plotly et formule les recommandations metier.

1. Dashboard interactif — Profils de risque (Plotly)
2. Dashboard interactif — Metriques des modeles (Plotly)
3. Dashboard interactif — Importance des variables (Plotly)
4. Dashboard interactif — Courbes ROC (Plotly)
5. Dashboard interactif — Analyse individuelle (simulation)
6. Export HTML autonome (livrable partageable)
7. Recommandations strategiques
8. Limites et perspectives

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath('..')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, recall_score, precision_score, f1_score, accuracy_score

RANDOM_STATE = 42

# Chargement des donnees
df_raw = pd.read_csv('data/processed/tp1_student_risk_wrangled.csv')
df_raw['dropout_risk'] = df_raw['dropout_risk'].astype(bool)

df_model = pd.read_csv('data/processed/tp1_student_risk_model_ready.csv')
y = df_model['dropout_risk'].astype(int)
X = df_model.drop(columns=['dropout_risk'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
])
logistic.fit(X_train, y_train)
logistic_prob = logistic.predict_proba(X_test)[:, 1]
logistic_pred = logistic.predict(X_test)

rf = RandomForestClassifier(n_estimators=400, class_weight='balanced_subsample',
                            random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = rf.predict(X_test)

print('Donnees et modeles charges. Dashboard pret.')

## Dashboard 1 — Profils de risque par programme et semestre

In [ ]:
prog_risk = df_raw.groupby('program')['dropout_risk'].mean() * 100
prog_risk = prog_risk.sort_values(ascending=True)
sem_risk = df_raw.groupby('semester')['dropout_risk'].mean() * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Taux de risque par programme', 'Evolution par semestre')
)

fig.add_trace(
    go.Bar(
        y=prog_risk.index.tolist(),
        x=prog_risk.values,
        orientation='h',
        marker_color=['#188038' if v < 7 else '#F29900' if v < 10 else '#D93025' for v in prog_risk.values],
        text=[f'{v:.1f}%' for v in prog_risk.values],
        textposition='outside',
        name='Taux de risque'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=sem_risk.index.tolist(),
        y=sem_risk.values,
        mode='lines+markers+text',
        text=[f'{v:.1f}%' for v in sem_risk.values],
        textposition='top center',
        line=dict(color='#1A73E8', width=3),
        marker=dict(size=10, color='#1A73E8'),
        fill='tozeroy',
        fillcolor='rgba(26,115,232,0.1)',
        name='Risque par semestre'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='<b>Profils de risque — Programmes et Semestres</b>',
    title_font_size=16,
    height=420,
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.update_xaxes(title_text='Taux de risque (%)', row=1, col=1)
fig.update_xaxes(title_text='Semestre', tickvals=list(range(1, 7)), row=1, col=2)
fig.update_yaxes(title_text='Taux de risque (%)', row=1, col=2)
fig.show()

fig_profils = fig

## Dashboard 2 — Comparaison des metriques des modeles

In [ ]:
metrics_names = ['Accuracy', 'Precision', 'Rappel', 'F1', 'ROC-AUC']

def get_scores(y_true, y_pred, y_prob):
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0),
        f1_score(y_true, y_pred, zero_division=0),
        roc_auc_score(y_true, y_prob)
    ]

lr_scores = get_scores(y_test, logistic_pred, logistic_prob)
rf_scores = get_scores(y_test, rf_pred, rf_prob)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Logistic Regression',
    x=metrics_names, y=lr_scores,
    marker_color='#1A73E8',
    text=[f'{v:.3f}' for v in lr_scores],
    textposition='outside'
))
fig.add_trace(go.Bar(
    name='Random Forest',
    x=metrics_names, y=rf_scores,
    marker_color='#D93025',
    text=[f'{v:.3f}' for v in rf_scores],
    textposition='outside'
))

fig.update_layout(
    title_text='<b>Comparaison des metriques — Logistic Regression vs. Random Forest</b>',
    title_font_size=15,
    barmode='group',
    yaxis=dict(range=[0, 1.15], title='Score'),
    xaxis_title='Metrique',
    height=430,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(x=0.75, y=1.0)
)
fig.show()

print('=> Logistic Regression domine sur le Rappel (metrique prioritaire) et le ROC-AUC.')

fig_metriques = fig

## Dashboard 3 — Importance des variables (Top 10)

In [ ]:
fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=False).head(10)

fig = px.bar(
    fi[::-1],
    x='importance',
    y='feature',
    orientation='h',
    color='importance',
    color_continuous_scale='Blues',
    text='importance',
    title='<b>Top 10 des variables explicatives du risque etudiant (Random Forest)</b>'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(
    height=430,
    xaxis_title='Importance (Gini)',
    yaxis_title='',
    coloraxis_showscale=False,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.show()

fig_features = fig

## Dashboard 4 — Courbes ROC interactives

In [ ]:
fig = go.Figure()

for prob, label, color in [
    (logistic_prob, 'Logistic Regression', '#1A73E8'),
    (rf_prob, 'Random Forest', '#D93025')
]:
    fpr, tpr, thresholds = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{label} (AUC = {auc:.3f})',
        line=dict(color=color, width=2.5),
        hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>'
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Aleatoire (AUC = 0.500)',
    line=dict(color='#70757A', dash='dash', width=1.5)
))

fig.update_layout(
    title_text='<b>Courbes ROC — Comparaison des modeles</b>',
    title_font_size=15,
    xaxis_title='Taux de faux positifs (FPR)',
    yaxis_title='Taux de vrais positifs (TPR)',
    height=470,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(x=0.55, y=0.05)
)
fig.show()

fig_roc = fig

## Dashboard 5 — Simulation : score de risque individuel

In [ ]:
# Simulation d'un profil etudiant type pour montrer le score de risque
sample = X_test.head(20).copy()
sample['risk_score'] = (logistic_prob[:20] * 100).round(1)
sample['risk_label'] = ['A risque' if p >= 50 else 'Non a risque' for p in sample['risk_score']]
sample['student'] = [f'ETD-{i+1:03d}' for i in range(20)]

sample_disp = sample[['student', 'risk_score', 'risk_label']].sort_values('risk_score', ascending=False)

fig = px.bar(
    sample_disp,
    x='student',
    y='risk_score',
    color='risk_label',
    color_discrete_map={'A risque': '#D93025', 'Non a risque': '#188038'},
    text='risk_score',
    title='<b>Score de risque individuel — Simulation sur 20 etudiants (Logistic Regression)</b>'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(y=50, line_dash='dash', line_color='black',
              annotation_text="Seuil d'alerte (50%)", annotation_position='top right')
fig.update_layout(
    yaxis=dict(range=[0, 110], title='Score de risque (%)'),
    xaxis_title='Etudiant',
    height=430,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend_title='Statut predit'
)
fig.show()

fig_scores = fig

## 💾 Export — Dashboard HTML interactif autonome

Tous les graphiques sont assemblés dans un fichier HTML standalone, consultable hors notebook
et partageable avec les équipes pédagogiques sans installation Python.

In [ ]:
from pathlib import Path
from plotly.io import to_html
from IPython.display import HTML, display

ASSETS_DIR = Path('report/assets')
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
DASHBOARD_PATH = ASSETS_DIR / 'tp7_communication_dashboard.html'

CSS = (
    '*, *::before, *::after { box-sizing: border-box; }'
    'body { margin: 0; font-family: "Segoe UI", system-ui, sans-serif;'
    '       background: #f8fafc; color: #0f172a; }'
    '.page { max-width: 1440px; margin: 0 auto; padding: 32px 24px 60px; }'
    '.hero { background: linear-gradient(135deg, #0f766e 0%, #1d4ed8 100%);'
    '        color: white; padding: 36px 42px; border-radius: 22px; margin-bottom: 28px; }'
    '.hero h1 { font-size: 24px; margin: 0 0 12px; }'
    '.hero p  { margin: 0; line-height: 1.75; opacity: .92; font-size: 15px; }'
    '.hero .meta { margin-top: 14px; font-size: 13px; opacity: .72; }'
    '.card { background: white; border-radius: 18px; padding: 22px 26px 8px;'
    '        box-shadow: 0 4px 24px rgba(15,23,42,.07); margin-bottom: 22px; }'
    '.card h2 { font-size: 16px; margin: 0 0 6px; }'
    '.card .sub { font-size: 13px; color: #64748b; margin: 0 0 14px; line-height: 1.5; }'
    '.grid-2 { display: grid; grid-template-columns: 1fr 1fr; gap: 22px; }'
    '@media(max-width: 960px) { .grid-2 { grid-template-columns: 1fr; } }'
    '.rec { background: #eff6ff; border-left: 4px solid #1d4ed8;'
    '       padding: 20px 26px; border-radius: 12px; margin-bottom: 22px; }'
    '.rec h3 { margin: 0 0 10px; color: #1e3a8a; font-size: 16px; }'
    '.rec ul { margin: 0; padding-left: 18px; line-height: 2; color: #1e3a8a; font-size: 14px; }'
    '.limits { background: #fefce8; border-left: 4px solid #ca8a04;'
    '          padding: 20px 26px; border-radius: 12px; }'
    '.limits h3 { margin: 0 0 10px; color: #92400e; font-size: 16px; }'
    '.limits ul { margin: 0; padding-left: 18px; line-height: 2; color: #78350f; font-size: 14px; }'
)

SECTIONS = [
    (
        '\U0001f4ca Profils de risque \u2014 Programmes & Semestres',
        'Taux de d\u00e9crochage par fili\u00e8re et \u00e9volution par semestre.',
        fig_profils,
    ),
    (
        '\U0001f3c6 Comparaison des m\u00e9triques \u2014 LR vs Random Forest',
        'Logistic Regression domine sur le Rappel et le ROC-AUC, crit\u00e8res prioritaires pour la pr\u00e9vention.',
        fig_metriques,
    ),
    (
        '\U0001f50d Importance des variables explicatives (Top 10)',
        'Le contr\u00f4le continu, la moyenne ant\u00e9rieure et l\'engagement sont les principaux pr\u00e9dicteurs.',
        fig_features,
    ),
    (
        '\U0001f4c8 Courbes ROC \u2014 Performance des mod\u00e8les',
        'La R\u00e9gression Logistique (AUC ~0.90) discrimine mieux que le Random Forest (AUC ~0.84).',
        fig_roc,
    ),
    (
        '\U0001f6a8 Score de risque individuel \u2014 Simulation 20 \u00e9tudiants',
        'Chaque barre repr\u00e9sente le score de risque pr\u00e9dit. Seuil d\'alerte fix\u00e9 \u00e0 50\u202f%.',
        fig_scores,
    ),
]

# Assemblage du corps HTML
cards = []
for i, (title, desc, fig_obj) in enumerate(SECTIONS):
    plotlyjs = 'cdn' if i == 0 else False
    cards.append(
        '<div class="card">'
        f'<h2>{title}</h2>'
        f'<p class="sub">{desc}</p>'
        + to_html(fig_obj, include_plotlyjs=plotlyjs, full_html=False,
                  config={'responsive': True})
        + '</div>'
    )

# Grille 2 colonnes pour features + ROC
body = (
    cards[0]  # profils — pleine largeur
    + cards[1]  # metriques — pleine largeur
    + f'<div class="grid-2">{cards[2]}{cards[3]}</div>'  # features + ROC
    + cards[4]  # scores — pleine largeur
)

RECS = (
    '<div class="rec">'
    '<h3>\U0001f4a1 Recommandations op\u00e9rationnelles</h3>'
    '<ul>'
    '<li><strong>R1 \u2014 Alerte pr\u00e9coce :</strong> calculer le score LR chaque semaine '
    '\u00e0 partir de l\'assiduit\u00e9, des retards et de l\'activit\u00e9 LMS. Alerter d\u00e8s 50\u202f%.</li>'
    '<li><strong>R2 \u2014 Segmentation :</strong> distinguer profil acad\u00e9mique (tutorat), '
    'profil organisationnel (coaching) et profil social (soutien financier).</li>'
    '<li><strong>R3 \u2014 Explicabilit\u00e9 :</strong> accompagner chaque alerte des 2\u20113 '
    'variables contributives. La LR est interpr\u00e9table par les \u00e9quipes p\u00e9dagogiques.</li>'
    '<li><strong>R4 \u2014 Audit d\'\u00e9quit\u00e9 :</strong> surveiller que le mod\u00e8le ne '
    'discrimine pas syst\u00e9matiquement certains groupes (\u00e9ducation parentale, statut).</li>'
    '</ul>'
    '</div>'
)

LIMITS = (
    '<div class="limits">'
    '<h3>\u26a0\ufe0f Limites &amp; Perspectives</h3>'
    '<ul>'
    '<li><strong>Donn\u00e9es synth\u00e9tiques :</strong> performances non directement '
    'transposables \u00e0 un contexte r\u00e9el sans validation terrain.</li>'
    '<li><strong>D\u00e9s\u00e9quilibre de classes :</strong> ~8,5\u202f% de positifs \u2014 '
    'SMOTE ou seuil ajust\u00e9 am\u00e9lioreraient le recall.</li>'
    '<li><strong>Variables manquantes :</strong> facteurs extrascolaires '
    '(sant\u00e9, situation familiale) absents.</li>'
    '<li><strong>Validation temporelle :</strong> un split par cohorte serait plus rigoureux.</li>'
    '<li><strong>Perspectives :</strong> SHAP values, API temps r\u00e9el, '
    'alertes automatiques dans le SI p\u00e9dagogique.</li>'
    '</ul>'
    '</div>'
)

HTML_DOC = (
    '<!DOCTYPE html>\n'
    '<html lang="fr">\n'
    '<head>\n'
    '<meta charset="utf-8">\n'
    '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
    '<title>Dashboard \u2014 Risque de d\u00e9crochage \u00e9tudiant</title>\n'
    '<style>' + CSS + '</style>\n'
    '</head>\n'
    '<body><div class="page">\n'
    '<div class="hero">\n'
    '<h1>\U0001f4ca Dashboard \u2014 Pr\u00e9diction du risque de d\u00e9crochage \u00e9tudiant</h1>\n'
    '<p>Restitution interactive des r\u00e9sultats du pipeline ML entra\u00een\u00e9 sur '
    '1\u202f600 \u00e9tudiants issus de 4 fili\u00e8res. '
    'Destin\u00e9 aux \u00e9quipes p\u00e9dagogiques pour identifier et accompagner '
    'les profils \u00e0 risque avant qu\'il ne soit trop tard.</p>\n'
    '<p class="meta">Hugo RAGUIN &middot; Amine TALEB &middot; Elliot FIORESE'
    ' &nbsp;|&nbsp; IPSSI 2025-2026'
    ' &nbsp;|&nbsp; Mod\u00e8les\u202f: Logistic Regression &middot; Random Forest</p>\n'
    '</div>\n'
    + body
    + RECS
    + LIMITS
    + '\n</div></body></html>'
)

DASHBOARD_PATH.write_text(HTML_DOC, encoding='utf-8')
size_kb = DASHBOARD_PATH.stat().st_size // 1024

print(f'Dashboard sauvegarde : {DASHBOARD_PATH}')
print(f'Taille : {size_kb} Ko')

display(HTML(
    '<div style="background:#f0fdf4;border:1px solid #86efac;border-radius:10px;'
    'padding:14px 18px;margin-top:10px;">'
    '<strong>&#x2705; Dashboard HTML genere avec succes</strong><br>'
    f'<small>Fichier : <code>report/assets/tp7_communication_dashboard.html</code>'
    f' &middot; {size_kb}\u202fKo</small>'
    '</div>'
))

## Recommandations strategiques

Sur la base des resultats analytiques, quatre recommandations operationnelles sont formulees :

### R1 — Deployer un score d'alerte precoce
Calculer chaque semaine le score de risque (logistic regression) pour chaque etudiant a partir des donnees disponibles (assiduite, retards de remise, activite LMS, notes). Declencher une alerte a partir de **50 %** de score.

### R2 — Segmenter les interventions par profil
Ne pas traiter tous les etudiants a risque de facon uniforme. Distinguer :
- **Profil academique** (notes basses, forte pression) → tutorat pairs / soutien pedagogique
- **Profil organisationnel** (retards, faible LMS) → coaching methodo, rappels calendrier
- **Profil social** (non boursier, trajet long) → soutien financier, tiers-lieux de travail

### R3 — Privilegier l'explicabilite
Ne pas deployer de modele opaque. La regression logistique est recommandee car ses **coefficients sont interpretables** par les responsables pedagogiques. Chaque alerte doit etre accompagnee des 2-3 variables qui ont le plus contribue au score.

### R4 — Mettre en place un audit d'equite
Verifier regulierement que le modele ne discrimine pas systematiquement certains groupes (genre, origine, statut social). Les biais identifies dans l'EDA (education parentale) doivent etre surveilles.

## Limites de l'etude

| Limite | Impact | Mitigation possible |
|---|---|---|
| **Dataset synthetique** | Resultats non generalisables directement | Substituer un dataset reel anonymise |
| **Absence de dimension temporelle** | Pas d'alerte precoce par semaine | Construire des features dynamiques par periode |
| **Pas de gradient boosting** | La comparaison est incomplete (XGBoost/LightGBM non testes) | Ajouter ces modeles en Notebook 06 |
| **CNN sur images synthetiques** | Pas de valeur metier directe | Integrer de vrais documents pedagogiques scannes |
| **Biais socio-economiques** | Le modele peut amplifier les inegalites existantes | Audit equite systematique avant deploiement |

---

*Ce projet a ete realise dans le cadre du cours de Data Science — IPSSI 2025-2026*